In [26]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, AutoConfig

In [2]:
ds = Dataset.load_from_disk("D:/doc/my_project/lianxi/transformers-code-master/03-PEFT/data/alpaca_data_zh")

# train_data_file = "D:/doc/my_project/lianxi/transformers-code-master/03-PEFT/data/alpaca_data_zh"
# from datasets import load_dataset
# dataset = load_dataset('json', data_files={'train': 'data/train.json', 'test': 'data/dev.json'})

In [20]:
# #模型下载
# from modelscope import snapshot_download

model_path = 'G:/model_weights/models/model/bloom-1b4-zh'
# model_dir = snapshot_download('langboat/bloom-1b4-zh', local_dir=model_path)

In [32]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh", force_download=True)

tokenizer_config.json:   0%|          | 0.00/268 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/268 [00:00<?, ?B/s]

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6984bfcc-6cb6504b309f6d47187cfe78;8ec55e32-9542-4272-b017-fe7028348caf)

Entry Not Found for url: https://huggingface.co/api/models/Langboat/bloom-1b4-zh/tree/main/additional_chat_templates?recursive=false&expand=false.
additional_chat_templates does not exist on "main"

In [27]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)
ds[3]

{'output': '法国的首都是巴黎。', 'input': '', 'instruction': '法国的首都是什么？'}

In [28]:
print(config.vocab_size)
print(config.hidden_size)

250880
64


In [12]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [13]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [14]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [15]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [17]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-1b4-zh", low_cpu_mem_usage=True)

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

g:\software\anaconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lkjx0\.cache\huggingface\hub\models--Langboat--bloom-1b4-zh. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/5.59G [00:00<?, ?B/s]

In [25]:
model_path = "G:/model_weights/huggingface_model/models--Langboat--bloom-1b4-zh/"
model_path =  'G:/model_weights/models/model/bloom-1b4-zh'
model = AutoModelForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True)

RuntimeError: Error(s) in loading state_dict for Embedding:
	size mismatch for weight: copying a param with shape torch.Size([46145, 2048]) from checkpoint, the shape in current model is torch.Size([250880, 64]).

In [ ]:
sum()